# HMDA. Colorado, 2025


## Why do mortgage applications get abandoned?

### Business question

Around one in five US mortgage applications ends without a decision — the
applicant withdraws, or the file is closed because requested documents never
arrived. Each one is pure cost: the lender has pulled credit, ordered work and
paid staff time, and recovers nothing. The applicant has spent weeks and ended
up without a mortgage.

Unlike a denial, which is a decision, an abandonment is usually friction. Friction
can be fixed.

**Question:** among 2025 Colorado mortgage applications, are applications
submitted through third-party channels more likely to be abandoned before a
decision than those submitted directly to the lender — and does any gap survive
controlling for applicant demographics and loan characteristics?

### What this analysis is about

Nothing here was randomised. Borrowers choose whether to use a broker, and
brokers serve a different mix of borrowers than retail branches do. Any
difference found is an association, not a causal effect. The analysis quantifies
the gap and tests how much of it survives stratification; it does not establish
that channel causes abandonment.

### Data

CFPB HMDA Data Browser, 2025 loan-level records, Colorado, all institutions.
Field definitions from the FFIEC Filing Instructions Guide.

Note: HMDA contains no credit score. It is the strongest predictor in any lending
decision and it is absent by design, which limits what any HMDA-only analysis can
conclude.

## Structure

1. **Exploration** — column inventory, dtype diagnosis, missingness and sentinel values
2. **Cleaning** — defining the analysis set and the abandonment outcome
3. **Channel variable** — constructing a documented proxy from two HMDA fields
4. **Descriptive analysis** — abandonment by channel, by loan purpose, and stratified
5. **Hypothesis test** — chi-square test of independence and effect sizes
6. **Modelling** — logistic regression for an adjusted channel estimate
7. **Conclusions** — findings, limitations, and estimated cost

## Exploration 

In [51]:
import pandas as pd
import numpy as np 
from pathlib import Path

from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as smf


In [4]:
raw = Path("../data/raw")
processed = Path ("../data/processed")

file = raw / "state_CO.csv"
df_all = pd.read_csv(file, low_memory = False)

In [5]:
df_all.shape

(295125, 99)

In [7]:
df_all.dtypes.value_counts()

int64      40
float64    32
object     27
Name: count, dtype: int64

In [8]:
def profile_objects(df: pd.DataFrame, n_samples: int = 4) -> pd.DataFrame:
    """Summarise object-dtype columns to spot miscast numerics.

    Returns one row per object column with cardinality, null rate, and a
    sample of values. High cardinality plus numeric-looking samples usually
    means a numeric field contaminated by a sentinel string such as "Exempt".
    """
    objects = df.select_dtypes("object")

    return (
        pd.DataFrame({
            "n_unique": objects.nunique(),
            "percent_null": objects.isna().mean().mul(100).round(1),
            "samples": objects.apply(
                lambda s: list(s.dropna().unique()[:n_samples])
            ),
        })
        .sort_values("n_unique", ascending=False)
    )

In [10]:
profile_objects(df_all)

,n_unique,percent_null,samples
total_loan_costs,97153,51.9,"[0.00, 1690.00, 1790.00, 2104.00]"
origination_charges,61587,51.1,"[0.00, 900.00, 895.00, 13852.50]"
discount_points,52761,74.3,"[12057.50, 5550.00, 13416.82, 30.40]"
loan_to_value_ratio,48122,33.2,"[75, 85, 97, 74.13]"
lender_credits,25689,82.6,"[35.02, 287.50, 4346.40, 4268.88]"
rate_spread,16973,49.5,"[1.312, 0.334, 1.511, 1.01700]"
interest_rate,3554,32.2,"[7.25, 6.375, 5, 5.5]"
lei,1131,0.0,"[549300FGXN1K3HLB1R50, 7H6GLXDRUGQFU57RNE97, 5..."
total_points_and_fees,1011,98.6,"[1690.00, 1790.00, 0.00, Exempt]"
property_value,927,20.3,"[355000, 705000, 195000, 135000]"


In [12]:
df_all["applicant_age"].value_counts(dropna = False, normalize = True)§

applicant_age
35-44    0.228141
45-54    0.183173
25-34    0.177078
55-64    0.139856
8888     0.121023
65-74    0.087831
>74      0.038391
<25      0.024508
Name: proportion, dtype: float64

In [45]:
sentinel = df_all["applicant_age"].astype(str) == "8888"

In [47]:
age_known = np.where(sentinel, "age missing", "age present")

for col in ["business_or_commercial_purpose", "action_taken", "derived_sex", "derived_race", "loan_purpose"]:
    print(f"--- {col} ---")
    print(
        pd.crosstab(df_all[col], age_known, normalize="columns")
        .mul(100).round(1)
    )
    print()

--- business_or_commercial_purpose ---
col_0                           age missing  age present
business_or_commercial_purpose                          
1                                      12.2          3.6
2                                      86.8         96.0
1111                                    1.0          0.5

--- action_taken ---
col_0         age missing  age present
action_taken                          
1                     7.6         59.3
2                     0.3          3.8
3                     2.1         15.4
4                     1.4         15.7
5                     0.9          4.5
6                    87.7          0.6
7                     0.0          0.1
8                     0.0          0.6

--- derived_sex ---
col_0              age missing  age present
derived_sex                                
Female                     0.7         20.7
Joint                      1.0         39.2
Male                       1.1         32.4
Sex Not Available      

In [49]:
applications = df_all["action_taken"].isin([1, 2, 3, 4, 5])
still_missing = applications & sentinel

print(still_missing.sum())
print(f"{still_missing.mean() * 100:.1f}% of all rows")
print(f"{still_missing.sum() / applications.sum() * 100:.1f}% of applications")

4404
1.5% of all rows
1.7% of applications


### Cleaning and defining the analysis set

1. **Code labels.** Most HMDA fields are filed as integer codes. Mapping them to
   labels up front means every table and chart downstream is readable.
2. **Dropping non-applications.** `action_taken` codes 6, 7 and 8 are purchased
   loans and preapproval requests. Neither is an application with an applicant
   journey, so both are excluded.
3. **Dropping sentinel ages.** HMDA encodes "not provided" as `8888`. 88% of
   affected rows are purchased loans (already excluded); the 4,404 that remain
   are 1.7% of applications and are dropped so age stays clean downstream.
4. **Dropping `loan_purpose` 5 ("not applicable").** 230 rows with an 81.7%
   abandonment rate, far outside the range of every other category. Too few to
   analyse and too anomalous to pool with the rest; almost certainly a
   reporting artifact.

In [18]:
ACTION_LABELS = {
    1: "originated",
    2: "approved, not accepted",
    3: "denied",
    4: "withdrawn by applicant",
    5: "closed for incompleteness",
    6: "purchased loan",
    7: "preapproval denied",
    8: "preapproval approved, not accepted",
}

LOAN_PURPOSE_LABELS = {
    1: "home purchase",
    2: "home improvement",
    4: "other purpose",
    5: "not applicable",
    31: "refinancing",
    32: "cash-out refinancing",
}

LOAN_TYPE_LABELS = {
    1: "conventional",
    2: "FHA",
    3: "VA",
    4: "RHS or FSA",
}

OCCUPANCY_LABELS = {
    1: "principal residence",
    2: "second residence",
    3: "investment property",
}

LIEN_LABELS = {
    1: "first lien",
    2: "subordinate lien",
}

SUBMISSION_LABELS = {
    1: "submitted directly",
    2: "not submitted directly",
    3: "not applicable",
    1111: "exempt",
}

PAYABLE_LABELS = {
    1: "payable to institution",
    2: "not payable to institution",
    3: "not applicable",
    1111: "exempt",
}

CODE_LABELS = {
    "action_taken": ACTION_LABELS,
    "loan_purpose": LOAN_PURPOSE_LABELS,
    "loan_type": LOAN_TYPE_LABELS,
    "occupancy_type": OCCUPANCY_LABELS,
    "lien_status": LIEN_LABELS,
    "submission_of_application": SUBMISSION_LABELS,
    "initially_payable_to_institution": PAYABLE_LABELS,
}

APPLICATION_ACTIONS = [1, 2, 3, 4, 5]
ABANDONED_ACTIONS = [4, 5]
AGE_SENTINEL = "8888"

EXCLUDED_LOAN_PURPOSES = [5]

In [22]:
def assign_channel(df: pd.DataFrame) -> pd.Series:
    """Classify origination channel from submission route and initial creditor.

    HMDA has no channel field. This is a documented proxy built from two
    columns: whether the applicant submitted directly to the filing
    institution, and whether the loan was initially payable to it.
    Rows that are exempt or not applicable on either field are left as NaN
    and excluded from channel analysis.
    """
    submitted = df["submission_of_application"]
    payable = df["initially_payable_to_institution"]

    conditions = [
        (submitted == 1) & (payable == 1),
        (submitted == 2) & (payable == 1),
        payable == 2,
    ]
    channels = ["retail", "broker", "correspondent"]

    return pd.Series(
        np.select(conditions, channels, default=None),
        index=df.index,
        dtype="object",
    )

def build_analysis_set(df: pd.DataFrame) -> pd.DataFrame:
    """Filter raw HMDA records to analysable applications and add labels.

    Keeps 'action_taken' 1-5 (real applications), drops rows where the
    applicant age is the "not provided" sentinel, adds a binary 'abandoned'
    flag, and maps coded fields to readable labels in new '*_label' columns.
    The original coded columns are kept so results can be checked against
    the filing instructions.
    """
    out = (
        df.loc[df["action_taken"].isin(APPLICATION_ACTIONS)]
        .loc[lambda d: d["applicant_age"].astype(str) != AGE_SENTINEL]
        .loc[lambda d: ~d["loan_purpose"].isin(EXCLUDED_LOAN_PURPOSES)]
        .copy()
    )

    out["abandoned"] = out["action_taken"].isin(ABANDONED_ACTIONS).astype(int)

    for column, labels in CODE_LABELS.items():
        out[f"{column}_label"] = out[column].map(labels)

    out["channel"] = assign_channel(out)
    
    return out


df = build_analysis_set(df_all)

df["channel"] = assign_channel(df)

df["channel"].value_counts(dropna=False).to_frame("n").assign(
    pct=lambda d: d["n"].div(len(df)).mul(100).round(1)
)




print(f"raw records        {len(df_all):>8,}")
print(f"analysis set       {len(df):>8,}")
print(f"abandonment rate   {df['abandoned'].mean():>8.1%}")

raw records         295,125
analysis set        255,970
abandonment rate      20.5%


In [24]:
channel_crosstab = pd.crosstab(
    df["submission_of_application_label"],
    df["initially_payable_to_institution_label"],
    margins=True,
    margins_name="total",
)

(
    channel_crosstab.style
    .format(thousands=",")
    .background_gradient(cmap="Blues", axis=None)
    .set_caption("Submission route by whether the loan was payable to the filing institution")
)

initially_payable_to_institution_label,exempt,not applicable,not payable to institution,payable to institution,total
submission_of_application_label,,,,,
exempt,"1,197",0,0,0,"1,197"
not submitted directly,0,"3,143","12,243","41,232","56,618"
submitted directly,0,"7,000","5,576","185,579","198,155"
total,"1,197","10,143","17,819","226,811","255,970"


In [26]:
(
    df.dropna(subset=["channel"])
    .groupby("channel")["abandoned"]
    .agg(["mean", "count"])
    .assign(mean=lambda d: d["mean"].mul(100).round(1))
    .rename(columns={"mean": "abandonment_rate_pct", "count": "n"})
    .sort_values("abandonment_rate_pct", ascending=False)
)

,abandonment_rate_pct,n
channel,,
retail,21.1,185579
correspondent,13.6,17819
broker,12.9,41232


In [28]:
(
    pd.crosstab(
        df["channel"],
        df["loan_purpose_label"],
        normalize="index",
    )
    .mul(100)
    .round(1)
)

loan_purpose_label,cash-out refinancing,home improvement,home purchase,other purpose,refinancing
channel,,,,,
broker,13.5,2.3,60.9,4.1,19.3
correspondent,17.6,0.4,55.4,0.5,26.1
retail,17.5,11.1,40.4,14.9,16.0


In [30]:
(
    df.groupby("loan_purpose_label")["abandoned"]
    .agg(["mean", "count"])
    .assign(mean=lambda d: d["mean"].mul(100).round(1))
    .rename(columns={"mean": "abandonment_rate_pct", "count": "n"})
    .sort_values("abandonment_rate_pct", ascending=False)
)

,abandonment_rate_pct,n
loan_purpose_label,,
cash-out refinancing,25.7,43727
refinancing,22.3,44127
home purchase,19.4,114308
other purpose,19.3,31296
home improvement,14.2,22512


In [33]:
(
    pd.crosstab(
        df["loan_purpose_label"],
        df["channel"],
        values=df["abandoned"],
        aggfunc="mean",
    )
    .mul(100)
    .round(1)
)

channel,broker,correspondent,retail
loan_purpose_label,,,
cash-out refinancing,16.4,18.1,27.3
home improvement,3.9,29.2,13.6
home purchase,13.0,12.7,19.9
other purpose,5.9,4.5,18.7
refinancing,12.6,12.4,24.9


In [35]:
pd.crosstab(df["loan_purpose_label"], df["channel"])

channel,broker,correspondent,retail
loan_purpose_label,,,
cash-out refinancing,5555,3144,32567
home improvement,931,65,20534
home purchase,25091,9876,75056
other purpose,1703,88,27676
refinancing,7952,4646,29746


In [37]:
abandonment_by_purpose_channel = (
    pd.crosstab(
        df["loan_purpose_label"],
        df["channel"],
        values=df["abandoned"],
        aggfunc="mean",
    )
    .mul(100)
    .round(1)
)

counts_by_purpose_channel = pd.crosstab(
    df["loan_purpose_label"],
    df["channel"],
)

(
    abandonment_by_purpose_channel.style
    .format("{:.1f}%")
    .background_gradient(cmap="Reds", axis=None)
    .set_caption(
        "Table 3. Abandonment rate by origination channel, within loan purpose. "
        "Retail is higher than broker in every category, so the overall gap is "
        "not explained by differences in loan mix. Correspondent home improvement "
        "(n=65) is too small to interpret."
    )
)

channel,broker,correspondent,retail
loan_purpose_label,,,
cash-out refinancing,16.4%,18.1%,27.3%
home improvement,3.9%,29.2%,13.6%
home purchase,13.0%,12.7%,19.9%
other purpose,5.9%,4.5%,18.7%
refinancing,12.6%,12.4%,24.9%


In [39]:
retail_broker = df[df["channel"].isin(["retail", "broker"])]

contingency = pd.crosstab(
    retail_broker["channel"],
    retail_broker["abandoned"].map({0: "completed", 1: "abandoned"}),
)

contingency

abandoned,abandoned,completed
channel,,
broker,5326,35906
retail,39164,146415


In [41]:
chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"chi-square statistic  {chi2:>12,.1f}")
print(f"degrees of freedom    {dof:>12}")
print(f"p-value               {p_value:>12.3g}")
print()
print("Expected counts under independence:")
pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(0)

chi-square statistic       1,433.4
degrees of freedom               1
p-value                          0

Expected counts under independence:


abandoned,abandoned,completed
channel,,
broker,8088.0,33144.0
retail,36402.0,149177.0


In [43]:
broker_abandoned, broker_completed = contingency.loc["broker"]
retail_abandoned, retail_completed = contingency.loc["retail"]

n_broker = broker_abandoned + broker_completed
n_retail = retail_abandoned + retail_completed

rate_broker = broker_abandoned / n_broker
rate_retail = retail_abandoned / n_retail

odds_broker = broker_abandoned / broker_completed
odds_retail = retail_abandoned / retail_completed

cramers_v = np.sqrt(chi2 / contingency.to_numpy().sum())

ci_broker = proportion_confint(broker_abandoned, n_broker, method="wilson")
ci_retail = proportion_confint(retail_abandoned, n_retail, method="wilson")

effect_sizes = pd.DataFrame(
    {
        "metric": [
            "broker abandonment rate",
            "retail abandonment rate",
            "difference (percentage points)",
            "risk ratio (retail / broker)",
            "odds ratio (retail / broker)",
            "Cramér's V",
        ],
        "value": [
            f"{rate_broker:.1%}  (95% CI {ci_broker[0]:.1%}–{ci_broker[1]:.1%})",
            f"{rate_retail:.1%}  (95% CI {ci_retail[0]:.1%}–{ci_retail[1]:.1%})",
            f"{(rate_retail - rate_broker) * 100:.1f}",
            f"{rate_retail / rate_broker:.2f}",
            f"{odds_retail / odds_broker:.2f}",
            f"{cramers_v:.3f}",
        ],
    }
)

effect_sizes.style.hide(axis="index").set_caption(
    "Effect size: retail vs broker abandonment, 2025 Colorado applications"
)

metric,value
broker abandonment rate,12.9% (95% CI 12.6%–13.2%)
retail abandonment rate,21.1% (95% CI 20.9%–21.3%)
difference (percentage points),8.2
risk ratio (retail / broker),1.63
odds ratio (retail / broker),1.80
Cramér's V,0.079


In [53]:
model_vars = [
    "abandoned",
    "channel",
    "loan_purpose_label",
    "loan_type_label",
    "occupancy_type_label",
    "lien_status_label",
    "derived_sex",
    "derived_race",
    "derived_ethnicity",
    "applicant_age",
    "income",
    "loan_amount",
]

retail_broker_all = df.loc[df["channel"].isin(["retail", "broker"])]
model_df = retail_broker_all[model_vars].dropna().copy()

print(f"rows available  {len(retail_broker_all):>10,}")
print(f"rows in model   {len(model_df):>10,}")
print(f"dropped         {len(retail_broker_all) - len(model_df):>10,}")
print()
print("nulls by column:")
retail_broker_all[model_vars].isna().sum().sort_values(ascending=False).to_frame("n_null")

rows available     226,811
rows in model      215,529
dropped             11,282

nulls by column:


,n_null
income,11282
abandoned,0
channel,0
loan_purpose_label,0
loan_type_label,0
occupancy_type_label,0
lien_status_label,0
derived_sex,0
derived_race,0
derived_ethnicity,0


In [55]:
missing_income = retail_broker_all["income"].isna()

pd.DataFrame({
    "abandonment_rate_pct": [
        retail_broker_all.loc[missing_income, "abandoned"].mean() * 100,
        retail_broker_all.loc[~missing_income, "abandoned"].mean() * 100,
    ],
    "n": [missing_income.sum(), (~missing_income).sum()],
}, index=["income missing", "income present"]).round(1)

,abandonment_rate_pct,n
income missing,24.9,11282
income present,19.3,215529


In [57]:
pd.crosstab(
    retail_broker_all["channel"],
    missing_income.map({True: "income missing", False: "income present"}),
    normalize="index",
).mul(100).round(1)

income,income missing,income present
channel,,
broker,7.0,93.0
retail,4.5,95.5


## 6. Modelling

The descriptive analysis showed retail abandoning 8.2 points more than broker,
and stratifying by loan purpose showed the gap persisting in every category,
so the raw difference is not a composition effect on purpose alone. Logistic
regression extends that logic to all available covariates at once, producing a
single adjusted estimate of the channel effect.

**Why logistic.** The outcome is binary, so linear regression is inappropriate:
it would predict probabilities outside 0-1 and assume constant variance that a
0/1 outcome cannot have. Logistic regression models the log-odds, which is
unbounded, then transforms back to a probability. Exponentiated coefficients
are odds ratios, directly comparable to the unadjusted odds ratio of 1.80
found above.

**Covariate selection: avoiding leakage.** Several HMDA fields are populated
only for loans that closed — `interest_rate`, `rate_spread`, `total_loan_costs`,
`origination_charges`, `loan_term`. An abandoned application has no interest
rate because no loan was ever made. Including any of them would let the model
predict abandonment by detecting absence rather than learning anything about
applicants, so all are excluded. Every covariate used is known at the time of
application.

**Handling missing income.** Income is missing for 11,282 applications (5.0%).
These abandon at 24.9% against 19.3% for the rest, so the data is missing not
at random and dropping the rows would bias results toward lower abandonment.
Missingness is also uneven by channel (7.0% of broker applications, 4.5% of
retail). Income is therefore retained with a median fill plus an explicit
`income_missing` indicator, allowing the model to treat non-reporting as the
signal it evidently is.

**Transformations.** Income and loan amount are right-skewed with long upper
tails; both enter as `log1p` to prevent a small number of very large values
dominating the fit.

**Reference levels.** Categorical predictors are dummy-coded with the first
level alphabetically omitted as the reference. For channel this is `broker`,
so the channel coefficient is the effect of retail relative to broker.

In [64]:
model_df = retail_broker_all[model_vars].copy()

model_df["income_missing"] = model_df["income"].isna().astype(int)
model_df["income"] = model_df["income"].fillna(model_df["income"].median())

model_df = model_df.dropna()

print(f"rows in model  {len(model_df):>10,}")
print(f"income missing {model_df['income_missing'].sum():>10,}")

rows in model     226,811
income missing     11,282


In [66]:
model_df = retail_broker_all[model_vars].copy()

model_df["income_missing"] = model_df["income"].isna().astype(int)
model_df["income"] = model_df["income"].fillna(model_df["income"].median())

model_df = model_df.dropna()

print(f"rows in model  {len(model_df):>10,}")
print(f"income missing {model_df['income_missing'].sum():>10,}")

rows in model     226,811
income missing     11,282


In [68]:
formula = (
    "abandoned ~ channel + loan_purpose_label + loan_type_label "
    "+ occupancy_type_label + lien_status_label + derived_sex "
    "+ derived_race + derived_ethnicity + applicant_age "
    "+ np.log1p(income) + np.log1p(loan_amount) + income_missing"
)

logit_model = smf.logit(formula, data=model_df).fit()

print(logit_model.summary())

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


ValueError: array must not contain infs or NaNs

In [70]:
print(model_df["income"].describe())
print()
print(f"income <= -1:  {(model_df['income'] <= -1).sum():,}")
print(f"income < 0:    {(model_df['income'] < 0).sum():,}")
print(f"income == 0:   {(model_df['income'] == 0).sum():,}")

count    226811.000000
mean        201.268558
std        2394.383889
min      -77764.000000
25%          86.000000
50%         127.000000
75%         192.000000
max      772000.000000
Name: income, dtype: float64

income <= -1:  180
income < 0:    180
income == 0:   2,485


In [72]:
model_df["income_log"] = np.sign(model_df["income"]) * np.log1p(model_df["income"].abs())
model_df["loan_amount_log"] = np.log1p(model_df["loan_amount"])

assert np.isfinite(model_df["income_log"]).all(), "income_log has non-finite values"
assert np.isfinite(model_df["loan_amount_log"]).all(), "loan_amount_log has non-finite values"

model_df[["income", "income_log", "loan_amount", "loan_amount_log"]].describe().round(2)

,income,income_log,loan_amount,loan_amount_log
count,226811.00,226811.00,2.268110e+05,226811.00
mean,201.27,4.83,3.607988e+05,12.37
std,2394.38,0.91,5.387375e+05,1.04
min,-77764.00,-11.26,5.000000e+03,8.52
25%,86.00,4.47,1.150000e+05,11.65
50%,127.00,4.85,3.150000e+05,12.66
75%,192.00,5.26,4.950000e+05,13.11
max,772000.00,13.56,1.542950e+08,18.85


In [74]:
formula = (
    "abandoned ~ channel + loan_purpose_label + loan_type_label "
    "+ occupancy_type_label + lien_status_label + derived_sex "
    "+ derived_race + derived_ethnicity + applicant_age "
    "+ income_log + loan_amount_log + income_missing"
)

logit_model = smf.logit(formula, data=model_df).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.479618
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:              abandoned   No. Observations:               226811
Model:                          Logit   Df Residuals:                   226775
Method:                           MLE   Df Model:                           35
Date:                Tue, 03 Nov 2026   Pseudo R-squ.:                 0.03112
Time:                        13:04:51   Log-Likelihood:            -1.0878e+05
converged:                       True   LL-Null:                   -1.1228e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
Intercept     

In [76]:
odds_ratios = pd.DataFrame({
    "odds_ratio": np.exp(logit_model.params),
    "ci_low": np.exp(logit_model.conf_int()[0]),
    "ci_high": np.exp(logit_model.conf_int()[1]),
    "p_value": logit_model.pvalues,
}).round(3)

odds_ratios.sort_values("odds_ratio", ascending=False).style.set_caption(
    "Table 8. Adjusted odds ratios for abandonment (reference: broker channel)"
)

,odds_ratio,ci_low,ci_high,p_value
derived_ethnicity[T.Free Form Text Only],5.028000,1.915000,13.198000,0.001000
channel[T.retail],2.029000,1.965000,2.095000,0.000000
derived_sex[T.Sex Not Available],1.409000,1.337000,1.486000,0.000000
loan_type_label[T.RHS or FSA],1.366000,1.008000,1.851000,0.045000
derived_race[T.Free Form Text Only],1.214000,0.550000,2.677000,0.632000
loan_type_label[T.VA],1.206000,1.154000,1.260000,0.000000
applicant_age[T.>74],1.072000,1.014000,1.132000,0.014000
derived_sex[T.Male],1.056000,1.025000,1.088000,0.000000
loan_type_label[T.conventional],1.023000,0.989000,1.058000,0.195000
loan_amount_log,1.020000,1.004000,1.036000,0.015000


In [78]:
model_df["derived_ethnicity"].value_counts()

derived_ethnicity
Not Hispanic or Latino     153816
Ethnicity Not Available     32701
Hispanic or Latino          30717
Joint                        9559
Free Form Text Only            18
Name: count, dtype: int64